# Parte 3: Ingeniería de Características (Feature Engineering)

Una vez el dataset está limpio y libre de nulos o valores absurdos, este notebook se enfoca en la **creación de valor predictivo**.

## Objetivos
1. **Creación Temporal**: Extraer variables útiles como hora, día de la semana, mes o si es un día festivo (*holidays*).
2. **Creación Espacial/Distancia**: Calcular nuevas métricas de viaje (e.j., *haversine distance* extra, velocidad promedio estimada).
3. **Agrupaciones y Ratios**: Ratios entre features (ej: dinero/distancia) siempre sin caer en Leakage.
4. Establecer las bases para crear los pipelines formales (`ColumnTransformer`, escaladores, codificadores (`OneHotEncoder`, `TargetEncoder`)) antes del modelado.

### 1. Extracción de Features Temporales

In [ ]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import snowflake.connector

load_dotenv('../.env')
conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database=os.getenv('SNOWFLAKE_DATABASE'),
    role=os.getenv('SNOWFLAKE_ROLE', 'ACCOUNTADMIN')
)

# Traer muestra de la vista limpia de Snowflake (creada en NB 02)
# IMPORTANTE: Usamos SAMPLE (0.01) para extraer una pequeñísima muestra representativa (~100k filas).
# Extraer demasiados datos causará un Error de Memoria (OOM) en tu computadora local al hacer One-Hot.
query = 'SELECT * FROM ANALYTICS.REFINED.OBT_TRIPS_CLEAN SAMPLE (0.01);'
df = pd.read_sql(query, conn)

# ==========================================
# 1. ELIMINACIÓN DE DATA LEAKAGE
# ==========================================
leakage_cols = [
    'FARE_AMOUNT', 'EXTRA', 'MTA_TAX', 'TIP_AMOUNT', 'TOLLS_AMOUNT',
    'IMPROVEMENT_SURCHARGE', 'CONGESTION_SURCHARGE', 'AIRPORT_FEE', 'EHAIL_FEE',
    'TIP_PCT', 'DROPOFF_DATETIME', 'TRIP_DURATION_MIN', 'AVG_SPEED_MPH',
    'PAYMENT_TYPE', 'TRIP_TYPE', 'STORE_AND_FWD_FLAG', 'RUN_ID', 'INGESTED_AT_UTC'
]
df = df.drop(columns=[col for col in leakage_cols if col in df.columns], errors='ignore')

# ==========================================
# 2. INGENIERÍA TEMPORAL
# ==========================================
df['PICKUP_DATETIME'] = pd.to_datetime(df['PICKUP_DATETIME'])
df['pickup_hour'] = df['PICKUP_DATETIME'].dt.hour
df['pickup_dayofweek'] = df['PICKUP_DATETIME'].dt.dayofweek
df['is_weekend'] = df['pickup_dayofweek'].apply(lambda x: 1 if x >= 5 else 0)
df['pickup_month'] = df['PICKUP_DATETIME'].dt.month

df.head()

### 2. Extracción de Features Espaciales e Interacciones

In [ ]:
# ==========================================
# 3. INGENIERÍA ESPACIAL E INTERACCIONES
# ==========================================
# Como no tenemos lat/lon exactas en los datos recientes de TLC (solo Location IDs),
# crearemos interacciones cruzadas de rutas para capturar flujos específicos.
df['route_id'] = df['PU_LOCATION_ID'].astype(str) + '_' + df['DO_LOCATION_ID'].astype(str)

# Nota: Si quisieras una velocidad promedio ESTIMADA (sin Leakage), deberías cruzarlo
# con una tabla histórica promedio, pero por ahora solo usaremos la distancia directa.
print(f"Nuevas rutas únicas creadas: {df['route_id'].nunique()}")


### 3. Pipeline de Transformadores (Scikit-Learn Pre-processing)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np

# ==========================================
# 4. CONFIGURACIÓN DEL PREPROCESADOR
# ==========================================
# Definimos qué columnas numéricas y categóricas vamos a usar para el modelo final
num_features = ['PASSENGER_COUNT', 'TRIP_DISTANCE', 'pickup_hour', 'pickup_dayofweek', 'pickup_month']
cat_features = ['VENDOR_ID', 'RATE_CODE_ID', 'PU_LOCATION_ID', 'DO_LOCATION_ID', 'SERVICE_TYPE', 'is_weekend']

# StandardScaler para variables continuas
num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# OneHotEncoder para variables categóricas (usamos sparse_output=True para evitar MemoryError con grandes arrays)
cat_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

# Preprocesador Maestro
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ])

# ==========================================
# EJEMPLO DE APLICACIÓN EN LA MUESTRA
# ==========================================
# Como nuestra vista OBT_TRIPS_CLEAN ya está limitada a <= 2023, podemos ajustar (fit)
# el preprocesador aquí sin temor a Data Leakage de los años de Test.
X_sample_transformed = preprocessor.fit_transform(df)

print("Pipeline preprocesador estructurado.")
print(f"Dimensiones de la matriz resultante después del One-Hot Encoding: {X_sample_transformed.shape}")
print("Nota: La matriz resultante es una matriz dispersa (sparse matrix) para ahorrar RAM.")

### 4. Preparación y Exportación Final

In [ ]:
# ==========================================
# 5. PREPARACIÓN FINAL PARA EL MODELO
# ==========================================
# Exportar la lista de variables a usar para que el notebook 04 o src/models las consuma.
features_to_use = num_features + cat_features
target_col = 'TOTAL_AMOUNT'

print(f"El modelo predecirá: {target_col}")
print(f"Usando las variables predictivas: {features_to_use}")

# IMPORTANTE:
# La separación de los datos en Train (2015-2023), Val (2024) y Test (2025)
# NO es necesaria hacerla aquí manualmente en Pandas. 
# Ya tenemos las vistas materializadas OBT_TRIPS_TRAIN, OBT_TRIPS_VAL y OBT_TRIPS_TEST 
# en Snowflake. El script de entrenamiento simplemente las consumirá por separado.
